In [ ]:
import csv
import json
from pathlib import Path
from solve_kis import solve_kis

from siglip_encoder import SiglipEncoder
from search_clip import SiglipRetriever 
from caption_retriever import CaptionRetriever
from paths import KIS_ROOT, MAPPING_DIR, OBJECTS_DIR, safe_filename_component


In [ ]:
from bm25_retriever import BM25Retriever

In [ ]:
encoder = SiglipEncoder()
retriever = SiglipRetriever()

print("SigLIP OK")

In [ ]:
print("[*] Đang khởi tạo Caption Retriever (toàn bộ dữ liệu)...")

try:
    caption_retriever = CaptionRetriever()
    
    # Thống kê nhanh số video có trong mapping để dễ kiểm tra dữ liệu.
    unique_videos = sorted(
        {item["video_id"] for item in caption_retriever.mapping}
    ) if caption_retriever.mapping else []

    print(
        f"[*] caption_retriever sẵn sàng: "
        f"{caption_retriever.index.ntotal} vectors, "
        f"{len(unique_videos)} videos"
    )

except (FileNotFoundError, ValueError, RuntimeError) as e:
    print(f"[Lỗi] Không tìm thấy dữ liệu caption/index/mapping: {e}")
    print(
        "    Hãy đảm bảo đã có file trong các thư mục:\n"
        "    - caption_generator/Lxx/*.json\n"
        "    - index/Lxx/*.index\n"
        "    - caption_mapping/Lxx/*.json"
    )
    raise

In [ ]:
all_captions = caption_retriever.caption_data
print(f"[*] Đã thu thập được {len(all_captions)} frames caption để nạp vào BM25.")

In [ ]:
from object_filter import ObjectMetadataLookup
object_lookup = ObjectMetadataLookup(
    mapping_dir=MAPPING_DIR,
    objects_dir=OBJECTS_DIR,
)

print("Object Lookup OK")

In [ ]:
bm25_engine = BM25Retriever(all_captions)

In [ ]:
from vlm_reranker import VLMReranker

# Khởi tạo mô hình chấm điểm (Quá trình này tốn vài phút để tải checkpoint)
print("[*] Đang nạp Qwen2.5-VL Reranker lên GPU...")
vlm_engine = VLMReranker()
print("[*] VLM đã sẵn sàng!")

In [ ]:
query_path = KIS_ROOT / "structured_query.json"
if not query_path.is_file():
    raise FileNotFoundError(
        "Missing structured_query.json. Run: "
        "python structure_query.py \"<query>\" --query-id q_001"
    )
with query_path.open("r", encoding="utf-8") as query_file:
    structured_query = json.load(query_file)

print(json.dumps(structured_query, ensure_ascii=False, indent=2))

In [ ]:
print(f"\n[*] Bắt đầu giải quyết query: {structured_query.get('query_id')}")
final_ranked = solve_kis(
    structured_query=structured_query,
    visual_retriever=retriever,
    visual_encoder=encoder,
    text_bm25_retriever=bm25_engine,
    text_semantic_retriever=caption_retriever,
    object_lookup=object_lookup,
    vlm_reranker=vlm_engine,
    retrieval_top_k=100,
    rerank_top_k=20,
    result_top_k=100,
)

print("\n=== TOP 10 KẾT QUẢ CUỐI ===")
for i, result in enumerate(final_ranked[:10], 1):
    print(
        f"{i:02d}. video={result['video_id']} "
        f"frame={result['frame_id']} "
        f"score={result['final_score']:.4f}"
    )

In [ ]:
from IPython.display import Image as NotebookImage, display
import pandas as pd

from paths import video_image_dir

_pts_cache = {}

def frame_pts_time(video_id, frame_id):
    if video_id not in _pts_cache:
        _pts_cache[video_id] = pd.read_csv(MAPPING_DIR / f"{video_id}.csv")
    df = _pts_cache[video_id]
    rows = df[df["frame_idx"] == frame_id]
    if len(rows) == 0:
        return None
    return float(rows.iloc[0]["pts_time"])

def resolve_result_image(result):
    stored = result.get("image_path")
    if stored:
        candidate = Path(stored)
        if not candidate.is_absolute():
            candidate = KIS_ROOT / candidate
        if candidate.is_file():
            return candidate
    keyframe_n = result.get("keyframe_n")
    if keyframe_n is None:
        try:
            keyframe_n = object_lookup.get_keyframe_n(result["video_id"], result["frame_id"])
        except Exception:
            return None
    if keyframe_n is None:
        return None
    image_dir = video_image_dir(result["video_id"])
    for name in (f"{int(keyframe_n):03d}.jpg", f"{int(keyframe_n)}.jpg"):
        candidate = image_dir / name
        if candidate.is_file():
            return candidate
    return None

print("=== TOP 20 + ANH + LOCATION ===")
for rank, result in enumerate(final_ranked[:20], 1):
    video_id, frame_id = result["video_id"], result["frame_id"]
    pts = frame_pts_time(video_id, frame_id)
    location = f"{video_id} | frame {frame_id}" + (f" | t={pts:.2f}s" if pts is not None else " | t=?")
    print(f"{rank:02d}. {location}")
    print(f"    final={result.get('final_score', float('nan')):.4f} | vlm={result.get('vlm_raw_score')} | err={result.get('vlm_error')} | caption={(result.get('retrieval_text', '') or '')[:160]}")
    image_file = resolve_result_image(result)
    if image_file is not None:
        display(NotebookImage(filename=str(image_file), width=480))
    else:
        print("    [no local image]")


In [ ]:
print("=== PRE-VLM vs POST-VLM (TOP 20 theo thu hang cuoi) ===")
ordered_pre = sorted(final_ranked, key=lambda r: r["score"], reverse=True)
pre_rank_of = {id(r): i for i, r in enumerate(ordered_pre, 1)}
for post, result in enumerate(final_ranked[:20], 1):
    pre = pre_rank_of[id(result)]
    delta = pre - post
    arrow = f"+{delta}" if delta > 0 else (str(delta) if delta < 0 else "=")
    video_id, frame_id = result["video_id"], result["frame_id"]
    pts = frame_pts_time(video_id, frame_id)
    loc = f"{video_id} f{frame_id}" + (f" t={pts:.1f}s" if pts is not None else "")
    print(f"post={post:02d} pre={pre:02d} ({arrow:>3}) | {loc} | pre={result['score']:.4f} -> final={result['final_score']:.4f} (vlm={result.get('vlm_raw_score')})") 


In [ ]:
csv_output_dir = Path.home() / "Downloads"
if not csv_output_dir.is_dir():
    csv_output_dir = Path.home()

query_id = safe_filename_component(structured_query.get("query_id", "query"))
csv_output_path = csv_output_dir / f"{query_id}-kis.csv"

with csv_output_path.open(
    "w",
    encoding="utf-8",
    newline=""
 ) as csv_file:
    writer = csv.writer(csv_file, lineterminator="\n")
    for result in final_ranked[:100]:
        writer.writerow([result["video_id"], result["frame_id"]])

print(f"Saved results to {csv_output_path}")
